# Figure layout demo

Quick tour of `Row`/`Column`/`Overlay` (see `qewton/visualization/figure_plan.md`) -
arranging several plots side by side, stacked, or overlaid in one `Figure`,
composed with the existing facet mechanism.

Only 2D plots are used here (no WebGL needed) - if `fig.show()` doesn't render
inline, set the renderer explicitly:
```python
import plotly.io as pio
pio.renderers.default = "vscode"
```
or use `fig.save_html("out.html")` and open it in a browser.

In [1]:
import numpy as np
import qewton
from qewton.visualization import Figure, Row, Column, Overlay

## Building blocks

A couple of small helper functions - plain `LinePlot`s (sine waves) and a
faceted `ScatterPlot` - just to have something to arrange.

In [2]:
def line_plot(freq, n=200, title=None):
    Y = qewton.Variable("y", 1)
    x_axis = qewton.BatchAxes(n)
    t = np.linspace(0, 2 * np.pi, n)
    data = np.sin(freq * t)[:, None]
    config = qewton.DataConfiguration(x_axis, qewton.FeatureAxes(Y))
    return qewton.visualization.LinePlot(data, config, x=x_axis, y=Y, title=title)


def faceted_scatter(n_facets=3, n_samples=40, labels=None):
    X, Y = qewton.Variable("x", 1), qewton.Variable("y", 1)
    facet_axis = qewton.BatchAxes(n_facets)
    sample_axis = qewton.BatchAxes(n_samples)
    data = np.random.randn(n_facets, n_samples, 2)
    config = qewton.DataConfiguration(facet_axis, sample_axis, qewton.FeatureAxes(X * Y))
    facet = qewton.visualization.FacetSpec(facet_axis, orientation="col", labels=labels)
    plot = qewton.visualization.ScatterPlot(data, config, x=X, y=Y, controls=[facet])
    return plot, facet

## Row / Column / Overlay

`Row` places plots side by side, `Column` stacks them, `Overlay` draws them
into the same cell. They nest.

In [3]:
predicted = line_plot(1.0, title="Predicted")
exact = line_plot(1.05, title="Exact")

Figure(Row(predicted, exact)).show()

In [4]:
a, b = line_plot(1.0, title="a"), line_plot(2.0, title="b")

Figure(Column(a, b)).show()

In [5]:
# Overlay: several plots sharing one cell, one set of axes
wave_1 = line_plot(1.0, title="1x")
wave_2 = line_plot(3.0, title="3x")

Figure(Overlay(wave_1, wave_2)).show()

## Nesting and padding

`Row(Column(a, b), c)`: a 2-row column next to a single plot. The single
plot occupies the first cell of its row; the second cell is padded empty
rather than spanned (see figure_plan.md §4).

In [6]:
a, b, c = line_plot(1.0, title="a"), line_plot(2.0, title="b"), line_plot(3.0, title="c")

fig = Figure(Row(Column(a, b), c))
print("grid_shape:", fig.grid_shape())  # (2, 2) - c's second cell is empty
fig.show()

grid_shape: (2, 2)


## Panels and facets multiply

A `Row` of one faceted plot next to one plain plot: the plain plot's panel
still gets a full-size block (matching the facet extent), it just draws into
the first cell of its own block (figure_plan.md §5). Facet labels become part
of the subplot title, joined with the panel's own `plot.title` when both are
present (§7).

In [7]:
faceted, facet = faceted_scatter(3, labels=["low", "mid", "high"])
plain = line_plot(1.0, title="Reference")

fig = Figure(Row(faceted, plain))
print("grid_shape:", fig.grid_shape())  # (1, 6): 2 panels * 3 facet cols each
fig.show()

grid_shape: (1, 6)


## Editing a layout after construction

`add_plot()` appends a new row; `remove_plot()`/`replace_plot()` let you
adjust a returned layout in place instead of rebuilding it - useful once
`graph.visualize()` starts returning layouts directly.

In [8]:
fig = Figure()
fig.add_plot(line_plot(1.0, title="first"))
fig.add_plot(line_plot(2.0, title="second"))
print("grid_shape after two add_plot() calls:", fig.grid_shape())

replacement = line_plot(5.0, title="replacement")
fig.replace_plot(fig.panels[0][0].plots[0], replacement)
fig.show()

grid_shape after two add_plot() calls: (2, 1)
